In [2]:
"""Organise and filter relevant data from datasets to collect important info."""

# Import libraries
from pathlib import Path

import pandas as pd

In [ ]:
# Set relevant datasets
TEST_2020 = 'datasets/air_quality/2020s/2020_01.csv'
TEST_2019 = 'datasets/air_quality/pre_2020/2019_01.csv'
# Code number used for NO2
POLLUTANT_CODE = 8
# Station data file
STATION_FILE = 'datasets/stations/2022.csv'

In [4]:
def load_csv_file_as_data_f(path: str) -> pd.DataFrame:
    """Load CSV file.

    Args:
        path (str): Path to file

    Returns:
        pd.DataFrame: Pandas Dataframe.

    """
    return pd.read_csv(path, sep=',')

In [5]:
original_csv = load_csv_file_as_data_f(TEST_2020)
original_csv


,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,ESTACIO,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,H20,V20,H21,V21,H22,V22,H23,V23,H24,V24
0,8,Barcelona,19,Barcelona,4,7,2020,1,1,3.0,...,18.0,V,40.0,V,44.0,V,31.0,V,33.0,V
1,8,Barcelona,19,Barcelona,4,7,2020,1,2,13.0,...,18.0,V,10.0,V,3.0,V,57.0,V,NaN,N
2,8,Barcelona,19,Barcelona,4,7,2020,1,3,33.0,...,56.0,V,46.0,V,40.0,V,32.0,V,NaN,N
3,8,Barcelona,19,Barcelona,4,7,2020,1,4,12.0,...,5.0,V,7.0,V,2.0,V,3.0,V,NaN,N
4,8,Barcelona,19,Barcelona,4,7,2020,1,5,1.0,...,73.0,V,53.0,V,33.0,V,29.0,V,25.0,V
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1274,8,Barcelona,19,Barcelona,58,14,2020,1,27,60.0,...,60.0,V,58.0,V,60.0,V,61.0,V,NaN,N
1275,8,Barcelona,19,Barcelona,58,14,2020,1,28,58.0,...,62.0,V,58.0,V,61.0,V,65.0,V,NaN,N
1276,8,Barcelona,19,Barcelona,58,14,2020,1,29,70.0,...,24.0,V,42.0,V,49.0,V,50.0,V,51.0,V
1277,8,Barcelona,19,Barcelona,58,14,2020,1,30,36.0,...,46.0,V,56.0,V,61.0,V,60.0,V,56.0,V


In [6]:
def filter_pollutant(orig_df: pd.DataFrame) -> pd.DataFrame:
    """Filter to only NO2 relevant stats.

    Args:
        orig_df (pd.DataFrame): The original CSV dataframe.

    Returns:
        pd.DataFrame: The data filtered to NO2.

    """
    return orig_df[orig_df['CODI_CONTAMINANT'] == POLLUTANT_CODE].copy()

In [7]:
filtered_csv = filter_pollutant(original_csv)
filtered_csv

,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,ESTACIO,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,H20,V20,H21,V21,H22,V22,H23,V23,H24,V24
29,8,Barcelona,19,Barcelona,4,8,2020,1,1,27.0,...,64.0,V,66.0,V,63.0,V,56.0,V,48.0,V
30,8,Barcelona,19,Barcelona,4,8,2020,1,2,37.0,...,67.0,V,61.0,V,49.0,V,60.0,V,NaN,N
31,8,Barcelona,19,Barcelona,4,8,2020,1,3,53.0,...,43.0,V,40.0,V,39.0,V,38.0,V,NaN,N
32,8,Barcelona,19,Barcelona,4,8,2020,1,4,34.0,...,37.0,V,36.0,V,22.0,V,20.0,V,NaN,N
33,8,Barcelona,19,Barcelona,4,8,2020,1,5,13.0,...,73.0,V,69.0,V,61.0,V,54.0,V,49.0,V
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187,8,Barcelona,19,Barcelona,58,8,2020,1,27,4.0,...,8.0,V,11.0,V,9.0,V,8.0,V,NaN,N
1188,8,Barcelona,19,Barcelona,58,8,2020,1,28,9.0,...,10.0,V,11.0,V,7.0,V,4.0,V,NaN,N
1189,8,Barcelona,19,Barcelona,58,8,2020,1,29,3.0,...,46.0,V,27.0,V,21.0,V,18.0,V,15.0,V
1190,8,Barcelona,19,Barcelona,58,8,2020,1,30,28.0,...,15.0,V,7.0,V,5.0,V,5.0,V,8.0,V


In [8]:
def calc_pollute_daily_avg(filtered_df: pd.DataFrame) -> pd.DataFrame:
    """Calculate average NO2 per day.

    Args:
        filtered_df (pd.DataFrame): Data frame with only relevant stations.

    Returns:
        pd.DataFrame: Dataframe with average NO2 values.

    """
    # Get the hour columns with values for NO2.
    hour_cols = [col for col in filtered_df.columns if col.startswith('H')]
    # Get the V cols which define whethere a value was gathered.
    valid_cols = [col for col in filtered_df.columns if col.startswith('V')]
    # Choose to drop any invalid values listed as "N" in the V column.
    for hour, value in zip(hour_cols, valid_cols, strict=False):
        filtered_df.loc[filtered_df[value] == 'N', hour] = None

    # Compute mean of valid hourly values
    filtered_df['no2_daily_avg'] = filtered_df[hour_cols].mean(axis=1)

    return filtered_df

In [9]:
calc_pollute_daily_avg(filtered_csv)

,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,ESTACIO,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,V20,H21,V21,H22,V22,H23,V23,H24,V24,no2_daily_avg
29,8,Barcelona,19,Barcelona,4,8,2020,1,1,27.0,...,V,66.0,V,63.0,V,56.0,V,48.0,V,32.291667
30,8,Barcelona,19,Barcelona,4,8,2020,1,2,37.0,...,V,61.0,V,49.0,V,60.0,V,NaN,N,38.565217
31,8,Barcelona,19,Barcelona,4,8,2020,1,3,53.0,...,V,40.0,V,39.0,V,38.0,V,NaN,N,38.956522
32,8,Barcelona,19,Barcelona,4,8,2020,1,4,34.0,...,V,36.0,V,22.0,V,20.0,V,NaN,N,33.217391
33,8,Barcelona,19,Barcelona,4,8,2020,1,5,13.0,...,V,69.0,V,61.0,V,54.0,V,49.0,V,29.291667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187,8,Barcelona,19,Barcelona,58,8,2020,1,27,4.0,...,V,11.0,V,9.0,V,8.0,V,NaN,N,8.000000
1188,8,Barcelona,19,Barcelona,58,8,2020,1,28,9.0,...,V,11.0,V,7.0,V,4.0,V,NaN,N,6.782609
1189,8,Barcelona,19,Barcelona,58,8,2020,1,29,3.0,...,V,27.0,V,21.0,V,18.0,V,15.0,V,19.130435
1190,8,Barcelona,19,Barcelona,58,8,2020,1,30,28.0,...,V,7.0,V,5.0,V,5.0,V,8.0,V,9.333333


In [10]:
def create_date_column(filtered_df: pd.DataFrame) -> pd.DataFrame:
    """Generate a date column from given values.

    Args:
        filtered_df (pd.DataFrame): Dataframe with filtered stations.

    Returns:
        pd.DataFrame: Dataframe with a clear date column.

    """
    filtered_df['date'] = pd.to_datetime(
        {
            'year': filtered_df['ANY'],
            'month': filtered_df['MES'],
            'day': filtered_df['DIA'],
        }
    )
    return filtered_df


In [11]:
create_date_column(filtered_csv)

,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,ESTACIO,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,H21,V21,H22,V22,H23,V23,H24,V24,no2_daily_avg,date
29,8,Barcelona,19,Barcelona,4,8,2020,1,1,27.0,...,66.0,V,63.0,V,56.0,V,48.0,V,32.291667,2020-01-01
30,8,Barcelona,19,Barcelona,4,8,2020,1,2,37.0,...,61.0,V,49.0,V,60.0,V,NaN,N,38.565217,2020-01-02
31,8,Barcelona,19,Barcelona,4,8,2020,1,3,53.0,...,40.0,V,39.0,V,38.0,V,NaN,N,38.956522,2020-01-03
32,8,Barcelona,19,Barcelona,4,8,2020,1,4,34.0,...,36.0,V,22.0,V,20.0,V,NaN,N,33.217391,2020-01-04
33,8,Barcelona,19,Barcelona,4,8,2020,1,5,13.0,...,69.0,V,61.0,V,54.0,V,49.0,V,29.291667,2020-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187,8,Barcelona,19,Barcelona,58,8,2020,1,27,4.0,...,11.0,V,9.0,V,8.0,V,NaN,N,8.000000,2020-01-27
1188,8,Barcelona,19,Barcelona,58,8,2020,1,28,9.0,...,11.0,V,7.0,V,4.0,V,NaN,N,6.782609,2020-01-28
1189,8,Barcelona,19,Barcelona,58,8,2020,1,29,3.0,...,27.0,V,21.0,V,18.0,V,15.0,V,19.130435,2020-01-29
1190,8,Barcelona,19,Barcelona,58,8,2020,1,30,28.0,...,7.0,V,5.0,V,5.0,V,8.0,V,9.333333,2020-01-30


In [ ]:
def load_station_file(file_path: str) -> pd.DataFrame:
    """Load a station data file and builds its location.

    Args:
        file_path (string): path to the file

    Returns:
        pd.DataFrame: the list of station numbers and locations.

    """
    data_f = pd.read_csv(file_path, sep=',')

    # Filter for the pollutant
    data_f_no2 = data_f[data_f['Codi_Contaminant'] == POLLUTANT_CODE].copy()

    # Rename the columns
    data_f_no2 = data_f_no2.rename(
        columns={
            'Estacio': 'estacio',
            'nom_cabina': 'station_name',
            'Longitud': 'lon',
            'Latitud': 'lat',
        }
    )

    return data_f_no2[['estacio', 'station_name', 'lat', 'lon']]


In [13]:
station_info = load_station_file(STATION_FILE)
station_info

,estacio,station_name,lat,lon
0,50,Barcelona - Ciutadella,41.38640,2.1874
4,43,Barcelona - Eixample,41.38530,2.1538
11,44,Barcelona - Gràcia,41.39870,2.1534
18,57,Barcelona - Palau Reial,41.38750,2.1151
25,4,Barcelona - Poblenou,41.40390,2.2045
29,42,Barcelona - Sants,41.37880,2.1331
32,54,Barcelona - Vall Hebron,41.42610,2.1480
39,58,Barcelona - Observatori Fabra,41.41843,2.1239


In [14]:
def merge_station_data(
    data_f_pollute: pd.DataFrame, station_data: pd.DataFrame
) -> pd.DataFrame:
    """Merge station data with pollution data.

    Args:
        data_f_pollute (pd.DataFrame): Dataframe with pollution data.
        station_data (pd.DataFrame): Dataframe with station data.

    Returns:
        pd.DataFrame: Merged dataframe.

    """
    data_f_pollute = data_f_pollute.rename(columns={'ESTACIO': 'estacio'})

    return data_f_pollute.merge(station_data, on='estacio', how='left')


In [15]:
merge_station_data(filtered_csv, station_info)

,CODI_PROVINCIA,PROVINCIA,CODI_MUNICIPI,MUNICIPI,estacio,CODI_CONTAMINANT,ANY,MES,DIA,H01,...,V22,H23,V23,H24,V24,no2_daily_avg,date,station_name,lat,lon
0,8,Barcelona,19,Barcelona,4,8,2020,1,1,27.0,...,V,56.0,V,48.0,V,32.291667,2020-01-01,Barcelona - Poblenou,41.40390,2.2045
1,8,Barcelona,19,Barcelona,4,8,2020,1,2,37.0,...,V,60.0,V,NaN,N,38.565217,2020-01-02,Barcelona - Poblenou,41.40390,2.2045
2,8,Barcelona,19,Barcelona,4,8,2020,1,3,53.0,...,V,38.0,V,NaN,N,38.956522,2020-01-03,Barcelona - Poblenou,41.40390,2.2045
3,8,Barcelona,19,Barcelona,4,8,2020,1,4,34.0,...,V,20.0,V,NaN,N,33.217391,2020-01-04,Barcelona - Poblenou,41.40390,2.2045
4,8,Barcelona,19,Barcelona,4,8,2020,1,5,13.0,...,V,54.0,V,49.0,V,29.291667,2020-01-05,Barcelona - Poblenou,41.40390,2.2045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227,8,Barcelona,19,Barcelona,58,8,2020,1,27,4.0,...,V,8.0,V,NaN,N,8.000000,2020-01-27,Barcelona - Observatori Fabra,41.41843,2.1239
228,8,Barcelona,19,Barcelona,58,8,2020,1,28,9.0,...,V,4.0,V,NaN,N,6.782609,2020-01-28,Barcelona - Observatori Fabra,41.41843,2.1239
229,8,Barcelona,19,Barcelona,58,8,2020,1,29,3.0,...,V,18.0,V,15.0,V,19.130435,2020-01-29,Barcelona - Observatori Fabra,41.41843,2.1239
230,8,Barcelona,19,Barcelona,58,8,2020,1,30,28.0,...,V,5.0,V,8.0,V,9.333333,2020-01-30,Barcelona - Observatori Fabra,41.41843,2.1239


In [16]:
def process_pollution_file(
    file_path: Path | str, station_file: str = STATION_FILE
) -> pd.DataFrame:
    """Run the full NO2 processing pipeline for a single CSV file.

    Args:
        file_path (Path | str): Path to the air quality CSV file.
        station_file (str): Path to the station metadata file.

    Returns:
        pd.DataFrame: Merged dataframe with NO2 daily averages and station info.

    """
    orig_df = load_csv_file_as_data_f(file_path)
    filtered_df = filter_pollutant(orig_df)
    filtered_df = calc_pollute_daily_avg(filtered_df)
    filtered_df = create_date_column(filtered_df)
    station_df = load_station_file(station_file)

    return merge_station_data(filtered_df, station_df)


In [17]:
processed_df = process_pollution_file(TEST_2020, STATION_FILE)

In [ ]:
def clean_final_columns(final_data_f: pd.DataFrame) -> pd.DataFrame:
    """Clean columns to essential data.

    Args:
        final_data_f (pd.DataFrame): Data frames with all info added.

    Returns:
        pd.DataFrame: Cleaned data frame with only essential info.

    """
    final_data_f = final_data_f.rename(
        columns={'ANY': 'year', 'MES': 'month', 'DIA': 'day'}
    )

    # Step 2: Decide which columns to keep
    keep_cols = [
        'date',
        'year',
        'month',
        'day',
        'estacio',
        'station_name',
        'lat',
        'lon',
        'no2_daily_avg',
    ]

    # Keep only columns that exist.
    keep_cols = [col for col in keep_cols if col in final_data_f.columns]

    # Step 3: Return cleaned DataFrame
    return final_data_f[keep_cols].sort_values(['date']).reset_index(drop=True)


In [19]:
cleaned_df = clean_final_columns(processed_df)
cleaned_df

,date,year,month,day,estacio,station_name,lat,lon,no2_daily_avg
0,2020-01-01,2020,1,1,4,Barcelona - Poblenou,41.40390,2.2045,32.291667
1,2020-01-01,2020,1,1,43,Barcelona - Eixample,41.38530,2.1538,36.833333
2,2020-01-01,2020,1,1,42,Barcelona - Sants,41.37880,2.1331,25.000000
3,2020-01-01,2020,1,1,54,Barcelona - Vall Hebron,41.42610,2.1480,36.958333
4,2020-01-01,2020,1,1,58,Barcelona - Observatori Fabra,41.41843,2.1239,14.333333
...,...,...,...,...,...,...,...,...,...
227,2020-01-31,2020,1,31,43,Barcelona - Eixample,41.38530,2.1538,79.384615
228,2020-01-31,2020,1,31,42,Barcelona - Sants,41.37880,2.1331,44.652174
229,2020-01-31,2020,1,31,4,Barcelona - Poblenou,41.40390,2.2045,58.478261
230,2020-01-31,2020,1,31,57,Barcelona - Palau Reial,41.38750,2.1151,40.043478


In [20]:
def process_all_pollution_recursive(data_folder: str) -> pd.DataFrame:
    """Concatate all csvs into one data frame with clean data.

    Args:
        data_folder (str): Path to the folder with the data.

    Returns:
        pd.DataFrame: Data frame with al files' cleaned data.

    """
    files = sorted(Path(data_folder).rglob('*.csv'))
    results = []

    for file in files:
        df_single = process_pollution_file(file)
        df_clean = clean_final_columns(df_single)
        results.append(df_clean)

    return pd.concat(results, ignore_index=True)

In [21]:
total_data_set = process_all_pollution_recursive('datasets/air_quality/2020s/')
total_data_set

,date,year,month,day,estacio,station_name,lat,lon,no2_daily_avg
0,2020-01-01,2020,1,1,4,Barcelona - Poblenou,41.40390,2.2045,32.291667
1,2020-01-01,2020,1,1,43,Barcelona - Eixample,41.38530,2.1538,36.833333
2,2020-01-01,2020,1,1,42,Barcelona - Sants,41.37880,2.1331,25.000000
3,2020-01-01,2020,1,1,54,Barcelona - Vall Hebron,41.42610,2.1480,36.958333
4,2020-01-01,2020,1,1,58,Barcelona - Observatori Fabra,41.41843,2.1239,14.333333
...,...,...,...,...,...,...,...,...,...
14531,2024-12-31,2024,12,31,43,Barcelona - Eixample,41.38530,2.1538,51.708333
14532,2024-12-31,2024,12,31,42,Barcelona - Sants,41.37880,2.1331,38.666667
14533,2024-12-31,2024,12,31,4,Barcelona - Poblenou,41.40390,2.2045,45.666667
14534,2024-12-31,2024,12,31,57,Barcelona - Palau Reial,41.38750,2.1151,30.583333


In [38]:
# Grab test file for pre-2020 data strucutre
data_f_19 = pd.read_csv(TEST_2019)

In [ ]:
def clean_no2_value(value: str) -> float | None:
    """Clean the NO2 value from pre-2020 CSV format.

    Args:
        value (str): The no2 value to alter.

    Returns:
        float | None: Numeric NO2 value or None if missing.

    """
    # Check if the values are valid.
    if pd.isna(value):
        return None

    # Remove spaces
    value = str(value).strip()

    if value == '--':
        return None

    # Remove units
    value = value.replace('µg/m³', '')
    value = value.strip()

    # Convert to float
    try:
        return float(value)
    except ValueError:
        return None

In [ ]:
data_f_19['no2_value'] = data_f_19['valor_no2'].apply(clean_no2_value)
data_f_19['no2_value']

In [ ]:
def parse_2019_datetime(data_f: pd.DataFrame) -> pd.DataFrame:
    """Recreate a usable date column for the pre-202 data.

    Args:
        data_f (pd.DataFrame): Data frame to alter.

    Returns:
        pd.DataFrame: Data frame with usable datae added.
        
    """
    data_f = data_f.copy()

    # Convert 'generat' to a usable format
    data_f['datetime'] = pd.to_datetime(data_f['generat'], format='%d/%m/%Y %H:%M')

    # Extract the date
    data_f['date'] = data_f['datetime'].dt.date

    return data_f

In [48]:
data_f_19 = parse_2019_datetime(data_f_19)
data_f_19[['date']]


,date
0,2019-01-01
1,2019-01-01
2,2019-01-01
3,2019-01-01
4,2019-01-01
...,...
5899,2019-01-31
5900,2019-01-31
5901,2019-01-31
5902,2019-01-31


In [ ]:
def calculate_pre_2020_daily_avg(clean_date_f: pd.DataFrame) -> pd.DataFrame:
    """Calculate the pre-2020 average NO2.

    Args:
        clean_date_f (pd.DataFrame): Dataframe to use fo calcualting averages.

    Returns:
        pd.DataFrame: Data frame with NO2 averages.

    """
    clean_date_f = clean_date_f.copy()

    # DOn't use rows with missing values.
    clean_date_f = clean_date_f[clean_date_f['no2_value'].notna()]

    return (
        clean_date_f.groupby(['nom_cabina', 'latitud', 'longitud', 'date'])
        .agg(no2_daily_avg=('no2_value', 'mean'))
        .reset_index()
    )


In [50]:
calculate_pre_2020_daily_avg(data_f_19)

,nom_cabina,latitud,longitud,date,no2_daily_avg
0,Barcelona - Ciutadella,41.3864,2.1874,2019-01-01,38.708333
1,Barcelona - Ciutadella,41.3864,2.1874,2019-01-02,41.391304
2,Barcelona - Ciutadella,41.3864,2.1874,2019-01-03,47.285714
3,Barcelona - Ciutadella,41.3864,2.1874,2019-01-04,44.130435
4,Barcelona - Ciutadella,41.3864,2.1874,2019-01-05,47.809524
...,...,...,...,...,...
243,Barcelona - Vall Hebron,41.4261,2.1480,2019-01-27,25.708333
244,Barcelona - Vall Hebron,41.4261,2.1480,2019-01-28,19.750000
245,Barcelona - Vall Hebron,41.4261,2.1480,2019-01-29,27.916667
246,Barcelona - Vall Hebron,41.4261,2.1480,2019-01-30,18.791667
